In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers,models,regularizers
from tensorflow.keras.applications import ResNet50,EfficientNetB0
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping,CSVLogger
import os,json
import numpy as np
import pandas as pd

# **Loading the data for classification**

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Solar_PV system"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-3

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(factor = 0.1),
])

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR,"train"),
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    label_mode = "int",
    shuffle = True,
    seed = 42
).map(lambda x,y: (data_augmentation(x) / 255.0,y)).prefetch(tf.data.AUTOTUNE)

Found 1293 files belonging to 2 classes.


In [ ]:

val_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    seed=42,
    shuffle=False
).map(lambda x,y: (x/255.0, y)).prefetch(tf.data.AUTOTUNE)


Found 286 files belonging to 2 classes.


In [ ]:

test_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    seed=42,
    shuffle=False
).map(lambda x,y: (x/255.0, y)).prefetch(tf.data.AUTOTUNE)


Found 285 files belonging to 2 classes.


# **PV Classification Step**

## Resnet Classifier

In [ ]:

def build_resnet_classifier(trainable_backbone=False, dropout=0.5, l2=1e-4):
    base = ResNet50(include_top=False, weights='imagenet',
                    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    base.trainable = trainable_backbone
    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = layers.Rescaling(1./255)(inputs)
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(2, activation='softmax',
                           kernel_regularizer=regularizers.l2(l2))(x)
    model = models.Model(inputs, outputs, name="ResNet50_Classifier")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

resnet_model = build_resnet_classifier()
resnet_model.summary()


Model: "ResNet50_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_3 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │         4,098 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,591,810 (90.00 MB)

 Trainable params: 4,098 (16.01 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("Trained model file/custom_cnn_best.keras",
                                       monitor="val_accuracy", save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

callbacks[0].filepath = "Trained model file/resnet_best.keras"
resnet_model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)


Epoch 1/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.5329 - loss: 0.8096
Epoch 1: val_accuracy improved from -inf to 0.57343, saving model to Trained model file/resnet_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 297s 7s/step - accuracy: 0.5328 - loss: 0.8096 - val_accuracy: 0.5734 - val_loss: 0.6826
Epoch 2/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.5333 - loss: 0.7360
Epoch 2: val_accuracy did not improve from 0.57343
41/41 ━━━━━━━━━━━━━━━━━━━━ 348s 8s/step - accuracy: 0.5334 - loss: 0.7359 - val_accuracy: 0.5734 - val_loss: 0.6827
Epoch 3/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.4971 - loss: 0.7280
Epoch 3: val_accuracy did not improve from 0.57343
41/41 ━━━━━━━━━━━━━━━━━━━━ 290s 7s/step - accuracy: 0.4981 - loss: 0.7277 - val_accuracy: 0.5734 - val_loss: 0.6845
Epoch 4/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.5565 - loss: 0.6968
Epoch 4: val_accuracy did not improve from 0.57343
41/41 ━━━━━━━━━━━━━━━━━━━━ 283s 7s/step - accuracy: 0.5569 - 

## Custom CNN build from Scratch

In [ ]:
def build_custom_cnn(dropout=0.5, l2=1e-4):
    model = models.Sequential(name="Custom_CNN")
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same',
                            kernel_regularizer=regularizers.l2(l2),
                            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same',
                            kernel_regularizer=regularizers.l2(l2)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same',
                            kernel_regularizer=regularizers.l2(l2)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2)))
    model.add(layers.Dropout(dropout))
    model.add(layers.Dense(2, activation='softmax'))

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Custom_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,939,586 (49.36 MB)

 Trainable params: 12,939,138 (49.36 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("Trained model file/custom_cnn_best.keras",
                                       monitor="val_accuracy", save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]
custom_cnn.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)


Epoch 1/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.6041 - loss: 19.4293
Epoch 1: val_accuracy improved from -inf to 0.33916, saving model to Trained model file/custom_cnn_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 273s 6s/step - accuracy: 0.6057 - loss: 19.2249 - val_accuracy: 0.3392 - val_loss: 29.0839
Epoch 2/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7714 - loss: 2.0622
Epoch 2: val_accuracy improved from 0.33916 to 0.57692, saving model to Trained model file/custom_cnn_best.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 262s 6s/step - accuracy: 0.7713 - loss: 2.0500 - val_accuracy: 0.5769 - val_loss: 40.3400
Epoch 3/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7721 - loss: 0.6916
Epoch 3: val_accuracy did not improve from 0.57692
41/41 ━━━━━━━━━━━━━━━━━━━━ 270s 7s/step - accuracy: 0.7726 - loss: 0.6908 - val_accuracy: 0.5734 - val_loss: 25.7097
Epoch 4/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.8145 - loss: 0.5539
Epoch 4: val_accuracy did not improve fro

In [ ]:
best_cnn_model = tf.keras.models.load_model("Trained model file/custom_cnn_best.keras")
best_cnn_model.summary()


test_loss, test_acc = best_cnn_model.evaluate(test_data)
print(f"Best CNN Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")

Model: "Custom_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │    12,845,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,817,864 (148.08 MB)

 Trainable params: 12,939,138 (49.36 MB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 25,878,278 (98.72 MB)

9/9 ━━━━━━━━━━━━━━━━━━━━ 48s 5s/step - accuracy: 0.9365 - loss: 0.2424
Best CNN Test Accuracy: 0.8912, Test Loss: 0.3149


## Evaluation Metrics with Ensemble models (Custom CNN + Resnet)

In [ ]:
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report

best_cnn = tf.keras.models.load_model("Trained model file/custom_cnn_best.keras")
best_resnet = tf.keras.models.load_model("/content/Trained model file/resnet_best.keras")

def ensemble_predict(images, models):
    preds = [m.predict(images, verbose=0) for m in models]
    p_avg = np.mean(preds, axis=0)
    return np.argmax(p_avg, axis=1), p_avg

y_true, y_pred = [], []
for images, labels in test_data:
    preds_cls, _ = ensemble_predict(images, [best_cnn, best_resnet])
    y_true.extend(labels.numpy())
    y_pred.extend(preds_cls)

print("Ensemble Accuracy:", accuracy_score(y_true, y_pred))
print("Ensemble F1 Score:", f1_score(y_true, y_pred, average='binary'))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["No Solar", "Has Solar"]))

Ensemble Accuracy: 0.9052631578947369
Ensemble F1 Score: 0.9114754098360656
Confusion Matrix:
 [[119   2]
 [ 25 139]]
              precision    recall  f1-score   support

    No Solar       0.83      0.98      0.90       121
   Has Solar       0.99      0.85      0.91       164

    accuracy                           0.91       285
   macro avg       0.91      0.92      0.90       285
weighted avg       0.92      0.91      0.91       285



In [ ]:
import numpy as np

# Assume you already have best_cnn and best_resnet loaded
# and ensemble_predict(images, models) defined

def classify_with_buffer(images, models, buffer_size=1200):
    """
    Classify rooftop PV presence within buffer zones.
    If not present in 1200 sq. ft, fallback to 2400 sq. ft.
    Returns label + confidence score + buffer zone used.
    """
    preds_cls, preds_prob = ensemble_predict(images, models)

    results = []
    for i in range(len(preds_cls)):
        confidence = float(np.max(preds_prob[i]))
        label = int(preds_cls[i])  # 0 = No Solar, 1 = Has Solar
        buffer_used = buffer_size

        # If no solar detected in 1200 sq. ft, fallback to 2400 sq. ft
        if label == 0 and buffer_size == 1200:
            buffer_used = 2400
            # Re-run classification logic for expanded buffer zone
            # (Here we simulate by checking confidence threshold)
            if confidence < 0.6:
                label = 0  # still No Solar
            else:
                label = 1  # detected in larger buffer

        results.append({
            "predicted_label": "Has Solar" if label == 1 else "No Solar",
            "confidence": confidence,
            "buffer_zone_sqft": buffer_used
        })
    return results

# Example usage on test_data
buffer_results = []
for images, labels in test_data:
    buffer_results.extend(classify_with_buffer(images, [best_cnn, best_resnet], buffer_size=1200))

# Print sample outputs
for r in buffer_results[:5]:
    print(r)

{'predicted_label': 'Has Solar', 'confidence': 0.7092118263244629, 'buffer_zone_sqft': 2400}
{'predicted_label': 'Has Solar', 'confidence': 0.7092229127883911, 'buffer_zone_sqft': 2400}
{'predicted_label': 'Has Solar', 'confidence': 0.7090994715690613, 'buffer_zone_sqft': 2400}
{'predicted_label': 'Has Solar', 'confidence': 0.7087182402610779, 'buffer_zone_sqft': 2400}
{'predicted_label': 'Has Solar', 'confidence': 0.7090497612953186, 'buffer_zone_sqft': 2400}


# **Segmentation Step**

# **Loading Data for Segmentation**

## U-Net Segmentation zoom level

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

def conv_block(x, filters, l2=1e-4, dropout=0.3):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    return x

def encoder_block(x, filters):
    f = conv_block(x, filters)
    p = layers.MaxPooling2D((2,2))(f)
    return f, p

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, (2,2), strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(224,224,3)):
    inputs = layers.Input(input_shape)

    # Encoder
    f1, p1 = encoder_block(inputs, 64)
    f2, p2 = encoder_block(p1, 128)
    f3, p3 = encoder_block(p2, 256)
    f4, p4 = encoder_block(p3, 512)

    # Bottleneck
    bottleneck = conv_block(p4, 1024)

    # Decoder
    d1 = decoder_block(bottleneck, f4, 512)
    d2 = decoder_block(d1, f3, 256)
    d3 = decoder_block(d2, f2, 128)
    d4 = decoder_block(d3, f1, 64)

    outputs = layers.Conv2D(1, (1,1), activation="sigmoid")(d4)

    model = models.Model(inputs, outputs, name="U-Net")
    return model

unet = build_unet()
unet.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
             loss="binary_crossentropy",
             metrics=["accuracy"])
unet.summary()

Model: "U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 224, 224,  │      1,792 │ input_layer_14[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        256 │ conv2d_25[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 224, 224,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        256 │ conv2d_26[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 224, 224,  │          0 │ batch_normalizat… │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 112, 112,  │          0 │ dropout_16[0][0]  │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 112, 112,  │     73,856 │ max_pooling2d_10… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_27[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 112, 112,  │    147,584 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        512 │ conv2d_28[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Dropout)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_11    │ (None, 56, 56,    │          0 │ dropout_17[0][0]  │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_29 (Conv2D)  │ (None, 56, 56,    │    295,168 │ max_pooling2d_11… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_29[0][0]   │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, 56, 56,    │    590,080 │ batch_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │      1,024 │ conv2d_30[0][0] 

 Total params: 31,055,297 (118.47 MB)

 Trainable params: 31,043,521 (118.42 MB)

 Non-trainable params: 11,776 (46.00 KB)

In [ ]:
import glob, os, json
import numpy as np
import cv2

IMG_SIZE = (224,224)

# Convert JSON polygons → binary mask
def json_to_mask(json_path, img_shape=IMG_SIZE):
    with open(json_path, 'r') as f:
        data = json.load(f)

    mask = np.zeros(img_shape, dtype=np.uint8)

    # Example: LabelMe-style polygons
    for shape in data['shapes']:
        points = np.array(shape['points'], dtype=np.int32)
        cv2.fillPoly(mask, [points], 1)

    mask = np.expand_dims(mask, axis=-1)  # (224,224,1)
    return mask

def preprocess_image(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def preprocess_mask_from_json(json_path):
    mask = json_to_mask(json_path, img_shape=IMG_SIZE)
    mask = tf.convert_to_tensor(mask, dtype=tf.float32)
    return mask

def load_image_mask(img_path, json_path):
    img = preprocess_image(img_path)
    mask = preprocess_mask_from_json(json_path)
    return img, mask